# Lab 07-10 · Two-Port Networks

Companion to chapter [07-10](../../chapters/07-10-two-port-networks.md). Handout: [README.md](README.md).

Parameters are measured here by applying the open- and short-circuit tests to a nodal model of each circuit, so the chapter's formulas and conversion tables are being checked, not assumed. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import math
import shutil
import subprocess
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)
dB = lambda z: 20 * np.log10(np.abs(z))


def ngspice_values(path):
    out = subprocess.run(["ngspice", "-b", path], capture_output=True, text=True).stdout
    vals = {}
    for line in out.splitlines():
        parts = line.split("=")
        if len(parts) >= 2:
            try:
                vals[parts[0].strip()] = float(parts[1].split()[0])
            except (ValueError, IndexError):
                pass
    return vals


def nodal(Y, J):
    # solve Y V = J for node voltages (ground eliminated)
    return np.linalg.solve(np.asarray(Y, complex), np.asarray(J, complex))

ngspice available: True


## Part 1 — Measure, don't assume

Example 07-10.1's T network: 10 Ω and 20 Ω series arms, 30 Ω shunt; nodes p1, m, p2.

**🔮 Predict:** $z_{12}$ and $y_{12}$.

In [2]:
Ga, Gb, Gc = 1 / 10, 1 / 20, 1 / 30
# nodal matrix for nodes (p1, m, p2)
Yn = np.array([[Ga, -Ga, 0], [-Ga, Ga + Gb + Gc, -Gb], [0, -Gb, Gb]])
z = np.zeros((2, 2))
for col, node in ((0, 0), (1, 2)):           # 1 A into port 1, then into port 2, other port open
    J = np.zeros(3); J[node] = 1
    V = nodal(Yn, J).real
    z[0, col], z[1, col] = V[0], V[2]
y = np.zeros((2, 2))
for col, node in ((0, 0), (1, 2)):           # 1 V at one port, other port shorted: eliminate m
    Vp = [0, 0]; Vp[col] = 1
    Vm = (Ga * Vp[0] + Gb * Vp[1]) / (Ga + Gb + Gc)
    y[0, col] = Ga * (Vp[0] - Vm); y[1, col] = Gb * (Vp[1] - Vm)
print("z =", z.round(4)); print("y =", y.round(5)); print("y @ z =", (y @ z).round(10))
assert np.allclose(y @ z, np.eye(2))
if HAVE_NGSPICE:
    print("ngspice:", ngspice_values("ex1_params.cir"))

z = [[40. 30.]
 [30. 50.]]
y = [[ 0.04545 -0.02727]
 [-0.02727  0.03636]]
y @ z = [[ 1.  0.]
 [-0.  1.]]
ngspice: {'Doing analysis at TEMP': 27.0, 'y11': 0.04545455, 'y21': -0.0272727, 'y22': 0.03636364, 'zin': 34.0, 'av': 0.5882353, 'zout': 40.0}


## Part 2 — Conversions, and when they fail

**🔮 Predict:** which sets exist for a series inductor, a shunt capacitor and an ideal transformer?

In [3]:
def abcd_to_z(T):
    A, B, Cc, D = T.ravel()
    if abs(Cc) < 1e-15:
        return None
    return np.array([[A / Cc, (A * D - B * Cc) / Cc], [1 / Cc, D / Cc]])


def abcd_to_y(T):
    A, B, Cc, D = T.ravel()
    if abs(B) < 1e-15:
        return None
    return np.array([[D / B, -(A * D - B * Cc) / B], [-1 / B, A / B]])


w = 2 * math.pi * 1e6
for name, T in (("series 10 uH", np.array([[1, 1j * w * 10e-6], [0, 1]])),
                ("shunt 1 nF", np.array([[1, 0], [1j * w * 1e-9, 1]])),
                ("ideal 2:1 transformer", np.array([[2, 0], [0, 0.5]]))):
    print(f"{name:22s}: z exists: {abcd_to_z(T) is not None}, y exists: {abcd_to_y(T) is not None}, det = {np.linalg.det(T):.3f}")

series 10 uH          : z exists: False, y exists: True, det = 1.000+0.000j
shunt 1 nF            : z exists: True, y exists: False, det = 1.000+0.000j
ideal 2:1 transformer : z exists: False, y exists: False, det = 1.000


## Part 3 — Chains

**🔮 Predict:** the open-circuit gain of five unbuffered RC sections at $\omega RC = 1$.

In [4]:
R, Cap = 1e3, 100e-9
w = 1 / (R * Cap)
Tsec = np.array([[1 + 1j * w * R * Cap, R], [1j * w * Cap, 1]])
for n in range(1, 6):
    Tn = np.linalg.matrix_power(Tsec, n)
    print(f"{n} sections: unbuffered {dB(1 / Tn[0, 0]):7.2f} dB, buffered {n * dB(1 / (1 + 1j)):7.2f} dB")
assert abs(dB(1 / np.linalg.matrix_power(Tsec, 2)[0, 0]) + 9.542) < 1e-3
# Problem 07-10.7's L-match swept
L7, C7 = 20 / (2 * math.pi * 1e8), 1 / (2 * math.pi * 1e8 * 25)
f = np.linspace(50e6, 200e6, 301); rl = []
for ff in f:
    ww = 2 * math.pi * ff
    T = np.array([[1, 0], [1j * ww * C7, 1]]) @ np.array([[1, 1j * ww * L7], [0, 1]])
    zin = (T[0, 0] * 10 + T[0, 1]) / (T[1, 0] * 10 + T[1, 1])
    rl.append(min(-dB((zin - 50) / (zin + 50)), 60.0))  # cap the exact match at 60 dB
print(f"best return loss {max(rl):.0f} dB (capped; an exact match) at {f[int(np.argmax(rl))] / 1e6:.1f} MHz")

1 sections: unbuffered   -3.01 dB, buffered   -3.01 dB
2 sections: unbuffered   -9.54 dB, buffered   -6.02 dB
3 sections: unbuffered  -16.13 dB, buffered   -9.03 dB
4 sections: unbuffered  -22.50 dB, buffered  -12.04 dB
5 sections: unbuffered  -28.86 dB, buffered  -15.05 dB
best return loss 60 dB (capped; an exact match) at 100.0 MHz


## Part 4 — S-parameters

**🔮 Predict:** $\lvert S_{21}\rvert$ and $\lvert S_{11}\rvert$ of a 1 pF shunt capacitor at 10 GHz in 50 Ω.

In [5]:
Z0 = 50


def S_from_abcd(T):
    A, B, Cc, D = T.ravel()
    d = A + B / Z0 + Cc * Z0 + D
    return np.array([[(A + B / Z0 - Cc * Z0 - D) / d, 2 * (A * D - B * Cc) / d], [2 / d, (-A + B / Z0 - Cc * Z0 + D) / d]])


for f in (1e9, 5e9, 10e9):
    S = S_from_abcd(np.array([[1, 0], [1j * 2 * math.pi * f * 1e-12, 1]]))
    print(f"{f / 1e9:4.0f} GHz: |S21| = {abs(S[1, 0]):.4f}, |S11| = {abs(S[0, 0]):.4f}, reciprocal {np.isclose(S[0, 1], S[1, 0])}, |S11|^2+|S21|^2 = {abs(S[0, 0])**2 + abs(S[1, 0])**2:.6f}")
S10 = S_from_abcd(np.array([[1, 0], [1j * 2 * math.pi * 10e9 * 1e-12, 1]]))
assert abs(abs(S10[1, 0]) - 0.5370) < 1e-4
if HAVE_NGSPICE:
    print("ngspice:", ngspice_values("ex6_sparams.cir"))

   1 GHz: |S21| = 0.9879, |S11| = 0.1552, reciprocal True, |S11|^2+|S21|^2 = 1.000000
   5 GHz: |S21| = 0.7864, |S11| = 0.6177, reciprocal True, |S11|^2+|S21|^2 = 1.000000
  10 GHz: |S21| = 0.5370, |S11| = 0.8436, reciprocal True, |S11|^2+|S21|^2 = 1.000000
ngspice: {'Doing analysis at TEMP': 27.0, 's11a': 0.2, 's21a': 0.8, 's21b': 0.9878867, 's21bp': -8.92705, 's11b': 0.1551769, 's21c': 0.6903509, 's11c': 0.4392195, 's21g': 0.5370293, 's11g': 0.8435636}


## Your turn

1. A two-port has $\mathbf{z} = \begin{bmatrix}100 & 40\\40 & 60\end{bmatrix}$ Ω. Find its h-parameters.
2. What are the S-parameters of a perfect 50 Ω through-connection, and of an open circuit at port 1?
3. A cascade of a series 30 Ω resistor and a shunt 20 mS conductance: find its ABCD matrix and its input impedance with a 50 Ω load.

<details><summary>Answers</summary>

1. $\Delta_z$ = 6000 − 1600 = 4400 Ω²: $h_{11}$ = 4400/60 = **73.33 Ω**, $h_{12}$ = 40/60 = **0.6667**, $h_{21}$ = **−0.6667**, $h_{22}$ = 1/60 = **0.01667 S**.
2. Through: $S_{11} = S_{22}$ = 0, $S_{21} = S_{12}$ = 1. Open at port 1: $S_{11}$ = +1 (total reflection, in phase).
3. $\begin{bmatrix}1 & 30\\0 & 1\end{bmatrix}\begin{bmatrix}1 & 0\\0.02 & 1\end{bmatrix} = \begin{bmatrix}1.6 & 30\\0.02 & 1\end{bmatrix}$; $Z_{in} = (1.6\times50 + 30)/(0.02\times50 + 1)$ = **55 Ω** (check: 30 + 50 ∥ 50 = 55 ✓).

</details>

In [6]:
z = np.array([[100, 40], [40, 60]]); dz = np.linalg.det(z)
print("1)", round(dz / z[1, 1], 2), round(z[0, 1] / z[1, 1], 4), round(-z[1, 0] / z[1, 1], 4), round(1 / z[1, 1], 5))
T = np.array([[1, 30], [0, 1]]) @ np.array([[1, 0], [0.02, 1]])
print("3)", T, (T[0, 0] * 50 + T[0, 1]) / (T[1, 0] * 50 + T[1, 1]))
assert abs((T[0, 0] * 50 + T[0, 1]) / (T[1, 0] * 50 + T[1, 1]) - 55) < 1e-9

1) 73.33 0.6667 -0.6667 0.01667
3) [[1.6e+00 3.0e+01]
 [2.0e-02 1.0e+00]] 55.0
